In [11]:
from qamomile.optimization.aoa import AOAConverter

In [14]:
import jijmodeling as jm
import numpy as np
from qamomile.qiskit import QiskitTranspiler


@jm.Problem.define("Tiny coloring smoke test", sense=jm.ProblemSense.MINIMIZE)
def tiny_coloring(problem: jm.DecoratedProblem):
    num_nodes = problem.Length()
    num_colors = problem.Length()
    edges = problem.Graph()

    x = problem.BinaryVar(shape=(num_nodes, num_colors))

    problem += jm.sum(x[u, color] * x[v, color] for (u, v) in edges for color in num_colors)
    problem += problem.Constraint(
        "one_color_per_node",
        (jm.sum(x[u, color] for color in num_colors) == 1 for u in num_nodes),
    )


instance = tiny_coloring.eval(
    {
        "num_nodes": 2,
        "num_colors": 2,
        "edges": [(0, 1)],
    }
)

converter = AOAConverter(instance)
converter.spin_model = converter.spin_model.normalize_by_abs_max()
hamiltonian = converter.get_cost_hamiltonian()

transpiler = QiskitTranspiler()
ring_executable = converter.transpile(transpiler, p=1, mixer="fully-connected", block_size=2)

pair_indices = np.array([[0, 1], [2, 3]], dtype=np.uint64)
explicit_executable = converter.transpile(
    transpiler,
    p=1,
    pair_indices=pair_indices,
)

assert converter.spin_model.num_bits == 4
assert len(hamiltonian.terms) > 0
assert ring_executable is not None
assert explicit_executable is not None

result = explicit_executable.sample(
    transpiler.executor(),
    shots=32,
    bindings={"gammas": [0.2], "betas": [0.3]},
).result()
decoded = converter.decode(result)

best_sample, best_energy, _ = decoded.lowest()
assert len(decoded.samples) > 0
assert set(best_sample.values()) <= {0, 1}
assert np.isfinite(best_energy)

print("AOAConverter smoke test passed.")
print("Best sample:", best_sample)
print("Best energy:", round(best_energy, 4))

AOAConverter smoke test passed.
Best sample: {0: 1, 1: 0, 2: 0, 3: 1}
Best energy: 0.0
